<div align="center">

# ASL

</div>

Imports

In [1]:
import sys
sys.path.insert(0, ".")

import fiddle as fdl
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from configs.experiments import cnn_optimizer_sweep
from src.datasets import SignLanguageMNIST
from src.helpers import (
    build_model,
    build_optimizer,
    make_data_loaders,
    plot_class_distribution,
    plot_experiment_comparison,
    plot_label_samples,
    plot_training_history,
    resolve_device,
    set_seed,
    train_model,
)
from src.models import MLP

<div align="center" style="color: purple">

## **Download dataset**

</div>

In [2]:
# Dane już pobrane — odkomentuj jeśli potrzebujesz pobrać od nowa
# !kaggle datasets download datamunge/sign-language-mnist

## Getting the data

In [3]:
# Dane już wypakowane — odkomentuj jeśli potrzebujesz wypakować od nowa
# import zipfile
# with zipfile.ZipFile("sign-language-mnist.zip", 'r') as zip_ref:
#     zip_ref.extractall("data/asl_mnist")

In [4]:
train_path = "data/asl_mnist/sign_mnist_train.csv"
test_path = "data/asl_mnist/sign_mnist_test.csv"

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

train_df.head()

Train shape: (27455, 785)
Test shape: (7172, 785)


,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,3,107,118,127,134,139,143,146,150,153,...,207,207,207,207,206,206,206,204,203,202
1,6,155,157,156,156,156,157,156,158,158,...,69,149,128,87,94,163,175,103,135,149
2,2,187,188,188,187,187,186,187,188,187,...,202,201,200,199,198,199,198,195,194,195
3,2,211,211,212,212,211,210,211,210,210,...,235,234,233,231,230,226,225,222,229,163
4,13,164,167,170,172,176,179,180,184,185,...,92,105,105,108,133,163,157,163,164,179


In [5]:
plot_class_distribution(train_df, title="Class distribution in training set")

In [6]:
plot_label_samples(
    train_df,
    samples_per_class=5,
    title="5 przykładów z każdej klasy",
)

## Trening

**Criterion** (funkcja straty) — mierzy jak bardzo przewidywania modelu różnią się od prawdziwych etykiet. Model stara się ją minimalizować. Dla klasyfikacji wieloklasowej standardowo używamy `CrossEntropyLoss`, która łączy w sobie `Softmax` + `NLLLoss`.

**Optimizer** — algorytm który aktualizuje wagi modelu na podstawie gradientów obliczonych przez kryterium. Porównanie:
- `SGD` — prosty, przewidywalny, wymaga ręcznego tuningu learning rate
- `Adam` — adaptacyjny learning rate dla każdego parametru, zazwyczaj szybciej zbiega

In [7]:
DEVICE = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Device: {DEVICE}")

train_dataset = SignLanguageMNIST("data/asl_mnist/sign_mnist_train.csv")
test_dataset = SignLanguageMNIST("data/asl_mnist/sign_mnist_test.csv")

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=0)

INPUT_SIZE = 28 * 28
HIDDEN_SIZE = 256
OUTPUT_SIZE = 24

model = MLP(INPUT_SIZE, HIDDEN_SIZE, OUTPUT_SIZE).to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

print(model)

Device: mps
MLP(
  (fc1): Linear(in_features=784, out_features=256, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=256, out_features=24, bias=True)
)


Funkcje `train_epoch`, `evaluate` i `train_model` są teraz w `src/helpers/training.py`, żeby notebook zawierał tylko przebieg eksperymentu.

In [8]:
EPOCHS = 4
history = train_model(
    model,
    train_loader,
    test_loader,
    criterion,
    optimizer,
    DEVICE,
    epochs=EPOCHS,
    flatten=True,
)

Epoch   1/4 | Train loss: 2.2214  acc: 0.3484 | Val loss: 1.6836  acc: 0.5100
Epoch   2/4 | Train loss: 1.3089  acc: 0.6082 | Val loss: 1.4041  acc: 0.5615
Epoch   3/4 | Train loss: 0.9956  acc: 0.7032 | Val loss: 1.1748  acc: 0.6509
Epoch   4/4 | Train loss: 0.8043  acc: 0.7628 | Val loss: 1.1378  acc: 0.6304


In [9]:
plot_training_history(history, title=f"MLP — {EPOCHS} epoki")

best_epoch = history["val_acc"].index(max(history["val_acc"])) + 1
print(f"Najlepszy val acc: {max(history['val_acc']):.4f} (epoka {best_epoch})")

Najlepszy val acc: 0.6509 (epoka 3)


## Model CNN

Zamiast spłaszczać obraz do wektora (jak MLP), CNN przesuwa małe filtry po obrazie i uczy się wykrywać lokalne wzorce (krawędzie, kształty palców). Każda warstwa widzi coraz bardziej abstrakcyjne cechy.

Architektura:
- **Conv → BatchNorm → ReLU → MaxPool** × 3 bloki (rosnąca liczba filtrów: 32→64→128)
- **Dropout** przed klasyfikatorem — regularyzacja, zmniejsza overfitting
- **FC → FC** — klasyfikator na końcu

Poniżej CNN jest trenowany z konfiguracji Fiddle z `configs/experiments/cnn_sweep.py`: różne learning rate i optymalizatory, po 4 epoki każdy.

In [10]:
cnn_configs = [fdl.build(config) for config in cnn_optimizer_sweep()]
cnn_histories = {}
cnn_results = []

for experiment in cnn_configs:
    print("=" * 80)
    print(f"Experiment: {experiment.name}")
    print(
        f"Optimizer: {experiment.optimizer.name}, "
        f"lr={experiment.optimizer.learning_rate}, "
        f"momentum={experiment.optimizer.momentum}, "
        f"weight_decay={experiment.optimizer.weight_decay}"
    )

    set_seed(experiment.seed)
    experiment_device = resolve_device(experiment.device)
    train_loader_cnn, test_loader_cnn = make_data_loaders(experiment.dataset)

    model_cnn = build_model(experiment.model).to(experiment_device)
    criterion_cnn = nn.CrossEntropyLoss()
    optimizer_cnn = build_optimizer(experiment.optimizer, model_cnn.parameters())

    total_params = sum(p.numel() for p in model_cnn.parameters())
    print(f"Device: {experiment_device}")
    print(f"Liczba parametrów CNN: {total_params:,}")

    history_cnn = train_model(
        model_cnn,
        train_loader_cnn,
        test_loader_cnn,
        criterion_cnn,
        optimizer_cnn,
        experiment_device,
        epochs=experiment.epochs,
    )

    best_val_acc = max(history_cnn["val_acc"])
    best_epoch = history_cnn["val_acc"].index(best_val_acc) + 1
    cnn_histories[experiment.name] = history_cnn
    cnn_results.append(
        {
            "name": experiment.name,
            "optimizer": experiment.optimizer.name,
            "learning_rate": experiment.optimizer.learning_rate,
            "momentum": experiment.optimizer.momentum,
            "weight_decay": experiment.optimizer.weight_decay,
            "epochs": experiment.epochs,
            "best_val_acc": best_val_acc,
            "best_epoch": best_epoch,
        }
    )

    plot_training_history(history_cnn, title=f"{experiment.name} — {experiment.epochs} epoki")
    print(f"Najlepszy val acc: {best_val_acc:.4f} (epoka {best_epoch})")

Experiment: cnn_adam_lr_1e_3
Optimizer: adam, lr=0.001, momentum=0.0, weight_decay=0.0
Device: mps
Liczba parametrów CNN: 394,456
Epoch   1/4 | Train loss: 0.3837  acc: 0.8870 | Val loss: 0.1492  acc: 0.9532
Epoch   2/4 | Train loss: 0.0241  acc: 0.9940 | Val loss: 0.1063  acc: 0.9590
Epoch   3/4 | Train loss: 0.0236  acc: 0.9932 | Val loss: 0.0926  acc: 0.9622
Epoch   4/4 | Train loss: 0.0098  acc: 0.9972 | Val loss: 0.0826  acc: 0.9679


Najlepszy val acc: 0.9679 (epoka 4)
Experiment: cnn_adam_lr_3e_4
Optimizer: adam, lr=0.0003, momentum=0.0, weight_decay=0.0
Device: mps
Liczba parametrów CNN: 394,456
Epoch   1/4 | Train loss: 0.6770  acc: 0.8212 | Val loss: 0.2007  acc: 0.9402
Epoch   2/4 | Train loss: 0.0431  acc: 0.9946 | Val loss: 0.1211  acc: 0.9689
Epoch   3/4 | Train loss: 0.0154  acc: 0.9986 | Val loss: 0.1005  acc: 0.9608
Epoch   4/4 | Train loss: 0.0115  acc: 0.9985 | Val loss: 0.1268  acc: 0.9582


Najlepszy val acc: 0.9689 (epoka 2)
Experiment: cnn_sgd_lr_1e_2
Optimizer: sgd, lr=0.01, momentum=0.0, weight_decay=0.0
Device: mps
Liczba parametrów CNN: 394,456
Epoch   1/4 | Train loss: 1.5888  acc: 0.5758 | Val loss: 0.5929  acc: 0.8799
Epoch   2/4 | Train loss: 0.3256  acc: 0.9286 | Val loss: 0.2696  acc: 0.9331
Epoch   3/4 | Train loss: 0.1346  acc: 0.9796 | Val loss: 0.1924  acc: 0.9499
Epoch   4/4 | Train loss: 0.0736  acc: 0.9909 | Val loss: 0.1475  acc: 0.9591


Najlepszy val acc: 0.9591 (epoka 4)
Experiment: cnn_sgd_momentum_lr_1e_2
Optimizer: sgd, lr=0.01, momentum=0.9, weight_decay=0.0
Device: mps
Liczba parametrów CNN: 394,456
Epoch   1/4 | Train loss: 0.5073  acc: 0.8445 | Val loss: 0.1680  acc: 0.9439
Epoch   2/4 | Train loss: 0.0244  acc: 0.9942 | Val loss: 0.0567  acc: 0.9780
Epoch   3/4 | Train loss: 0.0116  acc: 0.9976 | Val loss: 0.0433  acc: 0.9834
Epoch   4/4 | Train loss: 0.0097  acc: 0.9976 | Val loss: 0.0415  acc: 0.9842


Najlepszy val acc: 0.9842 (epoka 4)


In [11]:
cnn_results_df = pd.DataFrame(cnn_results).sort_values(
    "best_val_acc",
    ascending=False,
).reset_index(drop=True)

plot_experiment_comparison(
    cnn_results_df,
    metric="best_val_acc",
    title="CNN — porównanie konfiguracji",
)

cnn_results_df

,name,optimizer,learning_rate,momentum,weight_decay,epochs,best_val_acc,best_epoch
0,cnn_sgd_momentum_lr_1e_2,sgd,0.0100,0.9,0.0,4,0.984244,4
1,cnn_adam_lr_3e_4,adam,0.0003,0.0,0.0,4,0.968907,2
2,cnn_adam_lr_1e_3,adam,0.0010,0.0,0.0,4,0.967931,4
3,cnn_sgd_lr_1e_2,sgd,0.0100,0.0,0.0,4,0.959147,4
